In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "06-gateway/scaling-admission-cost/agentic-scaling-lab/notebooks/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 01 · The arithmetic of agent scale

**What you'll learn.** How to get from "100,000 conversations a day" to tokens per minute, turns in flight,
Provisioned Throughput units and dollars — in the two minutes a whiteboard gives you. The whole model is
`scalelab/capacity.py` (about 120 lines); read it once, then use it.

> **The one-minute version.** The system-design round grades envelope arithmetic. The sentence to say out loud:
> *"The unit of work is a turn; a turn is ~2.2 model calls of ~5k tokens; so 100k conversations a day is
> about 14 M input tokens a minute at peak, which is above the Flash tier baseline — that is the constraint
> I'll design around, not Cloud Run."*

In [ ]:
import sys, asyncio, inspect, random
sys.path[:0] = [".", ".."]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from nbutil import todo, check, acheck, latency_cdfs, timeline
from scalelab.clock import CLOCK
%matplotlib inline
pd.set_option("display.width", 140)
CLOCK.reset(0.02)   # 50x faster than real time

## 1. The chain of multiplications

conversations/day → conversations/s → turns/s → model calls/s → tokens/min. Every later number hangs off this.

In [ ]:
from scalelab.capacity import Scenario, plan, plan_markdown, cost_per_call, gsus_needed, pt_usd_per_million_burndown, PRICES
s = Scenario()          # the anchor scenario: Meridian Mobile, 100k conversations/day
p = plan(s)
pd.DataFrame(p["rates"]).round(2)

In [ ]:
tokens = pd.DataFrame(p["tokens"]).T
tokens[["input_tpm", "uncached_input_tpm", "output_tpm"]].div(1e6).round(2).rename(columns=lambda c: c + " (M)").assign(vs_baseline=tokens.vs_baseline.round(2))

In [ ]:
ax = tokens.input_tpm.div(1e6).plot.bar(figsize=(6, 3.5), rot=0, title="Input tokens per minute vs the 10 M TPM Flash tier-3 baseline")
ax.axhline(s.tpm_baseline / 1e6, ls="--", c="grey"); ax.set_ylabel("M tokens / min"); plt.tight_layout()

Peak (×3) is already 1.4× the baseline; the incident (×10) is 4.6×. Nothing you buy makes 46 M TPM appear in a
minute — the incident has to be *shaped* (notebook 03/04); the peak can be bought (Provisioned Throughput) or
reduced (fewer tokens per call).

## 2. Little's law: concurrency is latency × rate

Things in the system = arrival rate × time each spends in it. Turns in flight size orchestrator memory and model
concurrency; concurrent *sessions* (users thinking between turns) size open connections and hot state.

In [ ]:
conc = pd.DataFrame({k: v for k, v in p["concurrency"].items() if isinstance(v, dict)}).round(0)
print(conc)
print(f"\nthe 10 M TPM baseline sustains ≈ {p['concurrency']['max_turns_per_s_for_baseline']:.1f} turns/s "
      f"≈ {p['concurrency']['max_inflight_for_baseline']:.0f} turns in flight — the starting in-flight cap")

## 3. Cost per conversation

Cached input is billed at 10 %; a Flash-Lite call costs a quarter of a Flash call. Route and cache *first*, because
every later percentage applies to the new baseline.

In [ ]:
cost = pd.Series(p["cost"]).round(4); print(cost)
ax = cost[["per_conversation_no_cache", "per_conversation_cached", "per_conversation_routed_cached"]].plot.bar(
    figsize=(6, 3.5), rot=0, title="$ per conversation"); ax.set_ylabel("USD"); plt.tight_layout()

## 4. Provisioned Throughput

One GSU of Gemini 3.5 Flash delivers 675 *burndown* tokens/s (uncached input ×1, cached ×0.1, output ×6). PT is a
guarantee, not a discount, unless the term is long **and** the units stay busy.

In [ ]:
pt = p["pt"]
print("GSUs to carry the standard-model share:", pt["gsus"])
print("monthly cost of the average GSUs by term:", {k: round(v) for k, v in pt["monthly_usd_for_average_gsus"].items()})
pd.DataFrame({"$/M burndown": pt["usd_per_million_burndown"], "break-even utilisation": pt["breakeven_utilisation"]}).round(2)

## 5. What breaks first

In [ ]:
pd.DataFrame(p["breaks_first"]).round(2)

## Your turn — solutions

#### (a) Little's law

Turns in flight and concurrent sessions from rates and durations.

In [ ]:
def inflight_turns(turns_per_s, turn_seconds):
    return turns_per_s * turn_seconds

def concurrent_sessions(conversations_per_s, turns_per_conversation, turn_seconds, think_seconds):
    return conversations_per_s * turns_per_conversation * (turn_seconds + think_seconds)

In [ ]:
def _a():
    r = p["rates"]["peak"]
    assert abs(inflight_turns(r["turns_per_s"], s.turn_seconds) - 125) < 1
    assert abs(concurrent_sessions(r["conversations_per_s"], s.turns_per_conversation, s.turn_seconds, s.think_seconds) - 1375) < 5
check("a: Little's law", _a)

#### (b) GSUs from burndown weights

Reproduce `gsus_needed` from the price table: burndown = uncached×1 + cached×0.1 + output×burn.

In [ ]:
def gsus(model, calls_per_s, input_tokens, output_tokens, cached_tokens):
    inp, out, cached_price, gsu_tokens_per_s, output_burn = PRICES[model]
    burn = (input_tokens - cached_tokens) + 0.1 * cached_tokens + output_burn * output_tokens
    return calls_per_s * burn / gsu_tokens_per_s

In [ ]:
def _b():
    for calls in (1, 10, 29.8):
        assert abs(gsus("gemini-3.5-flash", calls, 5000, 350, 2700) - gsus_needed("gemini-3.5-flash", calls, 5000, 350, 2700)) < 1e-9
check("b: GSUs", _b)

#### (c) Break-even utilisation

At what utilisation does a GSU cost the same per burndown token as pay-as-you-go?

In [ ]:
def breakeven(model, term, input_tokens=5000, output_tokens=350, cached_tokens=2700):
    from scalelab.capacity import burndown_tokens
    paygo = cost_per_call(model, input_tokens, output_tokens, cached_tokens) / burndown_tokens(model, input_tokens, output_tokens, cached_tokens) * 1e6
    return pt_usd_per_million_burndown(model, term) / paygo

In [ ]:
def _c():
    assert abs(breakeven("gemini-3.5-flash", "1y") - 0.75) < 0.02
    assert breakeven("gemini-3.5-flash", "1m") > 1.0   # a monthly term is never cheaper than pay-as-you-go
check("c: break-even", _c)

#### (d) Re-plan for a batch pipeline

2 M documents/day, one call per document (3k in, 300 out, nothing cached), processed in an 8-hour window, no think time. Build the Scenario and name the binding constraint.

In [ ]:
def batch_scenario():
    return Scenario(conversations_per_day=2_000_000, turns_per_conversation=1, calls_per_turn=1, input_tokens=3000,
                    cached_tokens=0, output_tokens=300, peak_factor=3.0, incident_factor=1.0, turn_seconds=2.0,
                    think_seconds=0.0, lite_share=0.0)

BINDING_CONSTRAINT = "model throughput (tokens per minute) — nothing else is close"

In [ ]:
def _d():
    b = plan(batch_scenario())
    assert b["tokens"]["peak"]["vs_baseline"] > 1.0, "the window should exceed the baseline"
    assert b["concurrency"]["peak"]["concurrent_sessions"] == b["concurrency"]["peak"]["inflight_turns"], "no think time"
    assert BINDING_CONSTRAINT != "..."
    print("   peak input TPM:", round(b["tokens"]["peak"]["input_tpm"] / 1e6, 1), "M; GSUs:", b["pt"]["gsus"]["peak"])
check("d: batch re-plan", _d)

## Takeaways

- Do the chain: conversations → turns → calls → tokens/min. Say the peak TPM against the tier baseline.
- Little's law twice: in-flight turns (compute) and concurrent sessions (connections, hot state).
- Cost levers by integer factors: route, cache, compact. Output is 6× the price of input on Flash.
- PT: quote break-even utilisation before anyone buys a monthly term.
- Cloud Run is ~0.1 % of the model bill; what breaks first is the token budget, then the slowest downstream system.

## Verify before relying on it

Prices and model ids (5 Sep 2026 in `capacity.PRICES`), PayGo tier baselines (Flash 2/4/10 M TPM), PT GSU throughput and burndown weights, whether cached tokens count against the PayGo baseline.